# BERT and Encoder Models

Build a tiny bidirectional encoder and predict a masked token from its surrounding context. This is a local mechanics example, not a pretrained BERT model; it downloads no weights or data.

In [ ]:
import torch
from torch import nn

torch.manual_seed(31)
torch.set_num_threads(1)
print('CPU | masked-token encoder example')

In [ ]:
vocabulary_size, width = 8, 8
mask_id, target_id = 2, 5
tokens = torch.tensor([[1, 3, mask_id, 4, 0]])
positions = torch.arange(tokens.shape[1]).unsqueeze(0)
embedding = nn.Embedding(vocabulary_size, width)
position_embedding = nn.Embedding(8, width)
encoder_layer = nn.TransformerEncoderLayer(d_model=width, nhead=2, dim_feedforward=16,
                                            batch_first=True, dropout=0.0)
encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
prediction_head = nn.Linear(width, vocabulary_size)
padding_mask = tokens.eq(0)
hidden = encoder(embedding(tokens) + position_embedding(positions), src_key_padding_mask=padding_mask)
logits = prediction_head(hidden)
loss = nn.functional.cross_entropy(logits[:, 2], torch.tensor([target_id]))
loss.backward()
assert logits.shape == (1, 5, vocabulary_size)
assert torch.isfinite(loss)
assert embedding.weight.grad is not None
print('token logits:', tuple(logits.shape), '| masked-token loss:', round(loss.item(), 4))

The model can use tokens on both sides of the mask because there is no causal mask. A production BERT has a much larger vocabulary and pretrained weights; this random tiny model only demonstrates the data flow and loss.